# A3 – Initial-Conditions Ingestion Benchmark

Benchmarks end-to-end ingestion time for the Modal-side ingestion backends available in `aifs_modal`:

| # | Backend | Runs | Source | Credentials |
|---|---------|------|--------|-------------|
| 1 | **IFS from Brightband** | Modal (`us-east`) | Brightband ECMWF IFS on ArrayLake (zarr) | `ARRAYLAKE_API_TOKEN` |
| 2 | **ERA5 from ARCO** | Modal (`us-central1`) | Google Cloud Storage zarr (anonymous) | none |

All backends write into the shared Modal IC Volume — the comparison is purely how long the round trip `fetch → regrid (if needed) → zarr commit` takes.

Each benchmark ingests **two timesteps** (t-6h and t) — the minimum needed for a forecast.

```{note}
Local ingestion via `aifs_modal.ingest()` is no longer available. The `ifs-ekd` and
`era5-cds` sources run inline inside `run_forecast` when ICs are missing on the Volume.
```

> **Prerequisites:** `ARRAYLAKE_API_TOKEN` for Brightband; a Modal workspace for ARCO.

In [ ]:
import datetime as dt
import time

import matplotlib.pyplot as plt
import pandas as pd

import aifs_modal

## Configuration

Two consecutive 6-hourly analysis dates (t-6h and t).  Each backend fetches
~94 global variables, regrids from 0.25° lat/lon to N320, and writes two zarr
groups into the target bucket.

Set `STORAGE_BUCKET` to a bucket you can write to, and the `PREFIX_*`
values to throwaway key prefixes — this notebook will write and commit real
data there.

In [ ]:
# Brightband source repo (ECMWF IFS initial conditions on ArrayLake)
SOURCE_AL_REPO = "martibosch/ecmwf-ifs-hres-ics-open"

_today_00z = dt.datetime.now(dt.UTC).replace(hour=0, minute=0, second=0, microsecond=0)

# IFS sources (ifs-arraylake): 2 days ago — safely within all rolling windows
_ifs_end = _today_00z - dt.timedelta(days=2)
IFS_DATE_START = (_ifs_end - dt.timedelta(hours=6)).isoformat()
IFS_DATE_END = _ifs_end.isoformat()

# ERA5 sources (era5-arco): 7 days ago — past the ~5-day publication lag
_era5_end = _today_00z - dt.timedelta(days=7)
ERA5_DATE_START = (_era5_end - dt.timedelta(hours=6)).isoformat()
ERA5_DATE_END = _era5_end.isoformat()

print(f"IFS  dates : {IFS_DATE_START} → {IFS_DATE_END}")
print(f"ERA5 dates : {ERA5_DATE_START} → {ERA5_DATE_END}")

## 1. IFS initial conditions from CDS (local)

Fetches ECMWF open-data GRIB from AWS, regrids to N320, and writes the
zarr group to the target bucket.  No credentials needed.

In [ ]:
# IFS from open-data (ifs-ekd) runs inline inside run_forecast when ICs are missing.
# It is not available as a standalone ingestion function.

## 2. ERA5 from CDS (local)

Submits CDS API requests (surface, soil, pressure-level), waits for the
jobs to complete on the CDS side, downloads GRIB files, regrids to N320,
and writes to the bucket.  Latency is dominated by the CDS queue and
download time over the public internet.

In [ ]:
# ERA5 from CDS (era5-cds) runs inline inside run_forecast when ICs are missing.
# Local ingestion via aifs_modal.ingest() is no longer available.

## 3. IFS initial conditions from Brightband (local)

Reads zarr data directly from the Brightband ECMWF IFS dataset on the
Earthmover ArrayLake marketplace, maps variable names to the AIFS convention,
regrids from 0.25° lat/lon to N320, and writes to the target bucket.
Requires an authenticated ``arraylake.Client``.

In [ ]:
t0 = time.perf_counter()
with aifs_modal.app.run():
    aifs_modal.ingest_ifs_arraylake.remote(
        IFS_DATE_START,
        IFS_DATE_END,
        source_repo=SOURCE_AL_REPO,
    )
ifs_al_elapsed = time.perf_counter() - t0

print(f"IFS from Brightband (Modal us-east): ingested in {ifs_al_elapsed:.1f} s")

## 4. ERA5 from ARCO (Modal, `us-central1`)

Runs on Modal in the `us-central1` region, co-located with the public
ARCO-ERA5 bucket on GCS.  A single Modal container fetches and writes both
dates sequentially.  The measured time includes Modal container cold-start
and the final icechunk commit — i.e. the full wall-clock cost.

In [ ]:
with aifs_modal.app.run():
    t0 = time.perf_counter()
    aifs_modal.ingest_era5_arco.remote(
        ERA5_DATE_START,
        ERA5_DATE_END,
    )
    era5_arco_elapsed = time.perf_counter() - t0

print(f"ERA5 from ARCO (Modal us-central1): ingested in {era5_arco_elapsed:.1f} s")

## Comparison

In [ ]:
results = pd.DataFrame(
    {
        "backend": [
            "IFS from Brightband (Modal us-east)",
            "ERA5 from ARCO (Modal us-central1)",
        ],
        "elapsed_s": [
            ifs_al_elapsed,
            era5_arco_elapsed,
        ],
    }
).set_index("backend")
results["speedup"] = results["elapsed_s"].max() / results["elapsed_s"]
results.style.format({"elapsed_s": "{:.1f} s", "speedup": "{:.1f}×"})

In [ ]:
fig, ax = plt.subplots()
bars = ax.bar(results.index, results["elapsed_s"], width=0.5)
for bar, val in zip(bars, results["elapsed_s"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{val:.0f} s",
        ha="center",
        va="bottom",
    )
ax.set_ylabel("Elapsed time (s)")
ax.set_title(
    f"Initial-conditions ingestion time\n"
    f"IFS: {IFS_DATE_START} → {IFS_DATE_END}  |  "
    f"ERA5: {ERA5_DATE_START} → {ERA5_DATE_END}"
)
ax.set_ylim(0, max(results["elapsed_s"]) * 1.2)
fig.autofmt_xdate(rotation=15, ha="right")
fig.tight_layout()

## Summary

- **IFS from CDS (local)** — fetches ECMWF open-data GRIB from AWS. No credentials needed, regrids to N320. Good for on-demand single-date ingestion.
- **ERA5 from CDS (local)** — submits CDS API requests, waits for queue, downloads GRIB, regrids to N320. Covers the full ERA5 archive back to 1940 but queue latency dominates.
- **IFS from Brightband (local)** — zarr-to-zarr read with variable name mapping and regrid to N320 from the Earthmover ArrayLake marketplace dataset. Fastest local option when the source repo is available.
- **ERA5 from ARCO (Modal, `us-central1`)** — one Modal container handles both required dates sequentially, co-located with the GCS bucket. Includes cold-start and icechunk commit overhead.

## Cleanup

Remove all benchmark data from the bucket.

In [ ]:
# IC data is written to the Modal IC Volume and auto-cleaned by run_forecast.
# No S3 objects to delete here.